# MobileNet — Etapa 2: Implementação do MLP

Este notebook cobre a definição da arquitetura, o treinamento e a análise inicial do modelo **Multilayer Perceptron (MLP)** para classificação de faixa de preço de celulares.

---

## 1. Importações e Carregamento dos Dados

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

# Define o tema visual dos gráficos
sns.set_theme(style='whitegrid')

# Diretório de saída para as imagens geradas nesta etapa
IMAGES_DIR = '../images/training'
os.makedirs(IMAGES_DIR, exist_ok=True)

In [ ]:
# Leitura do dataset de treino
train = pd.read_csv('../data/train.csv')

# Separação das features e da variável alvo
X = train.drop(columns='faixa_preco')
y = train['faixa_preco']

# Divisão treino/validação com estratificação (mesma configuração da Etapa 1)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Normalização: fit apenas no treino para evitar data leakage
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)

print(f'Treino:    {X_train.shape}')
print(f'Validação: {X_val.shape}')

---
## 2. Arquitetura do MLP

O **Perceptron Multicamadas (MLP)** é uma rede neural feedforward composta por uma camada de entrada, uma ou mais camadas ocultas e uma camada de saída. As conexões são unidirecionais (da entrada para a saída) e cada neurônio se conecta a todos os neurônios da camada seguinte.

### 2.1 Estrutura da rede

```
Camada de Entrada   Camada Oculta 1   Camada Oculta 2   Camada de Saída
  (20 features)  →  (128 neurônios) →  (64 neurônios)  →   (4 classes)
```

### 2.2 Propagação direta (*Forward Pass*)

Em cada camada $l$, a saída é calculada aplicando a função de ativação sobre a combinação linear das entradas:

$$a^{(l)} = f\left(W^{(l)} \cdot a^{(l-1)} + b^{(l)}\right)$$

Onde:
- $W^{(l)}$ — matriz de pesos da camada $l$
- $b^{(l)}$ — vetor de bias da camada $l$
- $f(\cdot)$ — função de ativação
- $a^{(0)} = x$ — vetor de features de entrada

### 2.3 Função de ativação — ReLU

Aplicada nas camadas ocultas para introduzir não-linearidade. Sem ela, o MLP se reduziria a uma regressão linear:

$$\text{ReLU}(x) = \max(0,\, x)$$

A ReLU é preferida por evitar o problema do **gradiente que desaparece** (*vanishing gradient*), comum na função sigmoide.

### 2.4 Camada de saída — Softmax

Transforma os escores brutos da última camada em probabilidades para cada classe:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\displaystyle\sum_{j=1}^{K} e^{z_j}}, \quad \sum_{i=1}^{K} \text{softmax}(z_i) = 1$$

Onde $K = 4$ é o número de classes. A classe predita é aquela com maior probabilidade.

### 2.5 Função de perda — Entropia Cruzada

Mede a diferença entre a distribuição de probabilidade predita e os rótulos verdadeiros:

$$\mathcal{L} = -\sum_{i=1}^{K} y_i \log(\hat{y}_i)$$

Onde $y_i = 1$ se $i$ for a classe verdadeira e $0$ caso contrário (codificação *one-hot*). Quanto menor a perda, mais próximas as probabilidades preditas estão dos rótulos reais.

### 2.6 Retropropagação — Atualização dos pesos

Os pesos são ajustados iterativamente pelo **gradiente descendente**, minimizando a função de perda:

$$W^{(l)} \leftarrow W^{(l)} - \eta \cdot \frac{\partial \mathcal{L}}{\partial W^{(l)}}$$

Onde $\eta$ é a **taxa de aprendizado**. O otimizador **Adam** (Adaptive Moment Estimation) adapta $\eta$ individualmente para cada parâmetro com base nos momentos de primeira e segunda ordem dos gradientes:

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1)\nabla\mathcal{L}, \quad v_t = \beta_2 v_{t-1} + (1 - \beta_2)(\nabla\mathcal{L})^2$$

$$W \leftarrow W - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon}\hat{m}_t$$

Com $\beta_1 = 0{,}9$, $\beta_2 = 0{,}999$ e $\epsilon = 10^{-8}$ como valores padrão.

---
## 3. Treinamento do Modelo

In [ ]:
# Instanciação do MLPClassifier com a arquitetura definida
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64), # duas camadas ocultas: 128 e 64 neurônios
    activation='relu',            # função de ativação ReLU nas camadas ocultas
    solver='adam',                # otimizador Adam
    max_iter=500,                 # número máximo de iterações (épocas)
    random_state=42,              # semente para reprodutibilidade
    verbose=False,                # suprime logs de progresso
    early_stopping=True,          # interrompe o treino se não houver melhora
    validation_fraction=0.1,      # 10% do treino usado para monitorar early stopping
    n_iter_no_change=20           # paciência: para após 20 iterações sem melhora
)

# Treinamento do modelo
mlp.fit(X_train, y_train)

# Resultados iniciais
print(f'Iterações realizadas: {mlp.n_iter_}')
print(f'Acurácia no treino:   {mlp.score(X_train, y_train):.4f}')
print(f'Acurácia na validação:{mlp.score(X_val, y_val):.4f}')

---
## 4. Curva de Aprendizado

A curva de aprendizado acompanha o valor da função de perda (entropia cruzada) ao longo das iterações de treinamento. Um modelo bem treinado apresenta:

- **Perda de treino decrescente** — o modelo está aprendendo
- **Perda de validação acompanhando a de treino** — o modelo está generalizando bem
- **Estabilização de ambas as curvas** — convergência atingida

Se a perda de validação começar a subir enquanto a de treino diminui, o modelo está sofrendo **overfitting**.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

# Curva de perda no treino (armazenada automaticamente pelo MLPClassifier)
ax.plot(mlp.loss_curve_, label='Perda no treino', color='steelblue', linewidth=2)

# Curva de validação: validation_scores_ armazena acurácia, então invertemos para perda
if mlp.validation_scores_ is not None:
    perda_validacao = 1 - np.array(mlp.validation_scores_)
    ax.plot(perda_validacao, label='Perda na validação (1 - acurácia)',
            color='tomato', linewidth=2, linestyle='--')

ax.set_title('Curva de Aprendizado — Entropia Cruzada', fontsize=13)
ax.set_xlabel('Iteração')
ax.set_ylabel('Perda')
ax.legend()
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/01_learning_curve.png', dpi=150)
plt.show()

print(f'Perda final no treino: {mlp.loss_curve_[-1]:.4f}')

---
## 5. Resumo da Arquitetura

| Parâmetro               | Valor                     |
|-------------------------|---------------------------|
| Tamanho da entrada      | 20 features               |
| Camada oculta 1         | 128 neurônios             |
| Camada oculta 2         | 64 neurônios              |
| Camada de saída         | 4 classes                 |
| Ativação (ocultas)      | ReLU                      |
| Ativação (saída)        | Softmax                   |
| Função de perda         | Entropia cruzada          |
| Otimizador              | Adam                      |
| Máximo de iterações     | 500                       |
| Early stopping          | Sim (paciência = 20)      |
| Semente aleatória       | 42                        |

O modelo está pronto para avaliação detalhada na **Etapa 3**.